# UCSF-PDGM: deep-learning only (fine-tuned MobileNetV3 / EfficientNet-B0 / ResNet18), no traditional classifiers

**What this notebook does**
- Each of the 4 MRI slices (T1, T1c, T2, FLAIR) goes through ONE shared ImageNet-pretrained CNN that is **fine-tuned** end to end. The 4 feature vectors are concatenated.
- Age + sex go through a small branch. During training they are hidden for 30% of patients (modality dropout), so the same network learns to work with and without them.
- Class imbalance: class-weighted loss on the training part only (or a balanced sampler). Test folds are never balanced.
- Early stopping uses a validation part inside each training fold (never the test fold).
- Parts: MRI only, MRI + clinical, and MRI + clinical with age/sex hidden at test time (0/25/50/75/100%, only age, only sex), for binary (grade 2/3 vs 4) and 3-class.

**How to run**: use a free GPU (Kaggle or Colab: Runtime > GPU). On a CPU laptop set `MODELS = ["mobilenet_v3_small"]`, `IMG = 128`, `N_FOLDS = 3`, one task at a time. This code was tested only on random synthetic data (it runs end to end); it has NOT been run on the real dataset, so check the first fold's output before leaving it running.

**Honest note**: fine-tuning can help, but with 495 patients, one slice each, and only 99 / 43 / 56 minority patients, results can also stay close to the earlier models. Report whatever comes out, for all models, and do not pick the best model by its test score.

## Recovery after Colab disconnects
Mount Google Drive before running the settings cell. This version saves a checkpoint after every epoch and saves result rows incrementally to Drive. If Colab disconnects, reconnect, mount the same Google Drive, and rerun the notebook cells in order through the final run cell. It resumes unfinished folds and skips completed folds. The first run still needs internet to fetch the dataset and pretrained ImageNet weights; these are cached in Drive for later sessions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q datasets torch torchvision scikit-learn pandas openpyxl

In [ ]:
# ===== 1) SETTINGS (change things only here) =====
import os, copy, time, random, warnings, json
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (matthews_corrcoef, f1_score, balanced_accuracy_score, accuracy_score,
                             roc_auc_score, recall_score)
warnings.filterwarnings("ignore")

SYNTHETIC_TEST = False
SEEDS      = [42, 43, 44]      # final runs: all three seeds
N_FOLDS    = 5
IMG        = 160
EPOCHS     = 20
PATIENCE   = 5
BATCH      = 16
LR_BACKBONE, LR_HEAD = 1e-4, 1e-3
MODELS     = ["mobilenet_v3_small", "efficientnet_b0", "resnet18"]
TASKS      = ["binary", "3class"]
BALANCE    = "weighted_loss"  # "weighted_loss" | "sampler" | "none"
CLIN_DROP  = 0.3
MISSING_SCENARIOS = {"0% missing": 0.0, "25% missing": .25, "50% missing": .5, "75% missing": .75,
                     "100% missing": 1.0, "only age missing": "age", "only sex missing": "sex"}
PRETRAINED = not SYNTHETIC_TEST
TAG        = "deep_finetune"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# IMPORTANT: Cell 1 must mount Google Drive before this cell.
# Checkpoints, progress CSV, Hugging Face cache and pretrained weights persist in Drive.
if os.path.isdir("/content/drive/MyDrive"):
    SAVE_DIR = "/content/drive/MyDrive/UCSF_PDGM_checkpoints"
else:
    # Fallback only; files here may be lost when Colab runtime resets.
    SAVE_DIR = "/content/UCSF_PDGM_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(SAVE_DIR, "huggingface_cache")
os.environ["HF_HUB_CACHE"] = os.path.join(SAVE_DIR, "huggingface_cache", "hub")
torch.hub.set_dir(os.path.join(SAVE_DIR, "torch_hub"))
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(torch.hub.get_dir(), exist_ok=True)

PROGRESS_CSV = os.path.join(SAVE_DIR, f"deep_progress_{TAG}.csv")
RAW_CSV      = os.path.join(SAVE_DIR, f"deep_raw_{TAG}.csv")
SUMMARY_CSV  = os.path.join(SAVE_DIR, f"deep_summary_{TAG}.csv")
SUMMARY_XLSX = os.path.join(SAVE_DIR, f"deep_summary_{TAG}.xlsx")
DATA_CACHE   = os.path.join(SAVE_DIR, f"ucsf_pdgm_images_{IMG}.npz")
print("Persistent save directory:", SAVE_DIR)
print("Progress file:", PROGRESS_CSV)


device: cuda
Persistent save directory: /content/drive/MyDrive/UCSF_PDGM_checkpoints
Progress file: /content/drive/MyDrive/UCSF_PDGM_checkpoints/deep_progress_deep_finetune.csv


In [ ]:
# ===== 2) DATA: 4 MRI slices + age + sex + grade, loaded once into memory =====
def to_gray(im): return np.asarray(im.convert("L").resize((IMG, IMG)), dtype=np.uint8)

if SYNTHETIC_TEST:
    rng = np.random.default_rng(0); N = 60
    grade = rng.choice([2, 3, 4], size=N, p=[.2, .15, .65])
    X = rng.integers(0, 255, size=(N, 4, IMG, IMG), dtype=np.uint8)
    age = rng.integers(20, 85, size=N).astype(float); sex = rng.integers(0, 2, size=N).astype(float)
else:
    if os.path.exists(DATA_CACHE):
        print("Loading cached MRI/clinical arrays from Drive:", DATA_CACHE)
        with np.load(DATA_CACHE) as z:
            X, grade, age, sex = z["X"], z["grade"], z["age"], z["sex"]
    else:
        print("First run: downloading/loading dataset. It will be cached in Drive.")
        from datasets import load_dataset
        ds = load_dataset("MedOtter/UCSF-PDGM", split="test")
        X = np.stack([[to_gray(r[s]) for s in ["T1", "T1c", "T2", "FLAIR"]] for r in ds])
        grade = np.array(ds["who_cns_grade"]).astype(int)
        age = np.array(ds["age_at_mri"], dtype=float)
        sex = np.array([1.0 if s == "M" else 0.0 for s in ds["sex"]])
        np.savez_compressed(DATA_CACHE, X=X, grade=grade, age=age, sex=sex)
        print("Saved dataset cache to Drive.")

print("images", X.shape, "| grades", dict(zip(*np.unique(grade, return_counts=True))))

def make_labels(task):
    return (grade == 4).astype(int) if task == "binary" else np.searchsorted([2, 3, 4], grade)


First run: downloading/loading dataset. It will be cached in Drive.


README.md:   0%|          | 0.00/6.17k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 38.0MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/495 [00:00<?, ? examples/s]

Saved dataset cache to Drive.
images (495, 4, 160, 160) | grades {np.int64(2): np.int64(56), np.int64(3): np.int64(43), np.int64(4): np.int64(396)}


In [ ]:
# ===== 3) MODEL: shared pretrained backbone on each of the 4 sequences -> concatenate -> (+ age/sex branch) -> classifier =====
import torchvision.models as tvm
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def build_backbone(name):
    w = "DEFAULT" if PRETRAINED else None
    if name.startswith("mobilenet_v3"):
        m = getattr(tvm, name)(weights=w); m.classifier = nn.Identity()
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=w); m.classifier = nn.Identity()
    elif name == "resnet18":
        m = tvm.resnet18(weights=w); m.fc = nn.Identity()
    elif name == "swin_t":
        m = tvm.swin_t(weights=w); m.head = nn.Identity()
    else:
        raise ValueError(name)
    with torch.no_grad(): d = m.eval()(torch.zeros(2, 3, IMG, IMG)).shape[1]
    return m, d

class FusionNet(nn.Module):
    def __init__(self, name, n_classes, use_clinical):
        super().__init__()
        self.backbone, d = build_backbone(name); self.use_clinical = use_clinical
        self.mri = nn.Sequential(nn.Dropout(0.3), nn.Linear(4 * d, 128), nn.ReLU())
        self.clin = nn.Sequential(nn.Linear(4, 16), nn.ReLU())
        self.out = nn.Linear(128 + 16, n_classes)
    def forward(self, x, clin):                       # x: [B,4,H,W] uint8-scaled floats, clin: [B,4]
        B, S, H, W = x.shape
        z = x.reshape(B * S, 1, H, W).expand(-1, 3, -1, -1)
        z = (z - MEAN.to(x.device)) / STD.to(x.device)
        f = self.backbone(z).reshape(B, -1)
        c = self.clin(clin) if self.use_clinical else torch.zeros(B, 16, device=x.device)
        return self.out(torch.cat([self.mri(f), c], 1))

# clinical columns = [age (scaled), sex, age_missing, sex_missing]; hidden values are replaced by the training mean/mode
def clin_matrix(idx, stats, hide_age, hide_sex):
    a = np.where(hide_age, stats["age_mean"], age[idx]); s = np.where(hide_sex, stats["sex_mode"], sex[idx])
    return np.stack([(a - stats["age_mean"]) / stats["age_std"], s, hide_age.astype(float), hide_sex.astype(float)], 1).astype(np.float32)

def augment(x):                                       # light augmentation, training only
    flip = torch.rand(x.shape[0], 1, 1, 1) < 0.5; x = torch.where(flip, x.flip(-1), x)
    x = x * (1 + 0.1 * torch.randn(x.shape[0], 1, 1, 1)) + 0.05 * torch.randn(x.shape[0], 1, 1, 1)
    dx, dy = np.random.randint(-8, 9, 2); return torch.roll(x, (int(dy), int(dx)), (2, 3))


In [ ]:
# ===== 4) TRAIN ONE MODEL (with epoch checkpoints) + EVALUATE =====
def to_x(idx): return torch.tensor(X[idx], dtype=torch.float32) / 255.0

def predict(model, idx, clin):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(idx), 32):
            out.append(F.softmax(model(to_x(idx[i:i+32]).to(device), torch.tensor(clin[i:i+32]).to(device)), 1).cpu().numpy())
    return np.concatenate(out)

def score(y, proba, task):
    pred = proba.argmax(1); n = proba.shape[1]
    r = {"mcc": matthews_corrcoef(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
         "balanced_acc": balanced_accuracy_score(y, pred), "accuracy": accuracy_score(y, pred)}
    try: r["auc"] = roc_auc_score(y, proba[:, 1]) if n == 2 else roc_auc_score(y, proba, multi_class="ovr", average="macro")
    except ValueError: r["auc"] = np.nan
    for k, v in enumerate(recall_score(y, pred, average=None, labels=range(n), zero_division=0)): r[f"recall class {k}"] = v
    return r

def safe_torch_load(path):
    # Explicit weights_only=False is needed for optimizer/RNG state in newer PyTorch.
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def checkpoint_path(task, name, mode, seed, fold):
    key = f"{task}__{name}__{mode.replace(' ', '_')}__seed{seed}__fold{fold}"
    return os.path.join(SAVE_DIR, key + ".pt")

def train_one(name, y, tr, use_clinical, seed, task, mode, fold):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    n_classes = int(y.max()) + 1
    tr_in, va = train_test_split(tr, test_size=0.15, stratify=y[tr], random_state=seed)
    stats = dict(age_mean=age[tr_in].mean(), age_std=age[tr_in].std() + 1e-6,
                 sex_mode=float(np.round(sex[tr_in].mean())))
    model = FusionNet(name, n_classes, use_clinical).to(device)
    counts = np.bincount(y[tr_in], minlength=n_classes)
    w = torch.tensor(len(tr_in) / (n_classes * np.maximum(counts, 1)), dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=w if BALANCE == "weighted_loss" else None)
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": list(model.mri.parameters()) + list(model.clin.parameters()) + list(model.out.parameters()), "lr": LR_HEAD}], weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    sample_p = (1.0 / np.maximum(counts, 1))[y[tr_in]] if BALANCE == "sampler" else None
    best, best_state, bad, start_ep = -1, None, 0, 0
    ckpt = checkpoint_path(task, name, mode, seed, fold)

    if os.path.exists(ckpt):
        state = safe_torch_load(ckpt)
        model.load_state_dict(state["model_state"])
        opt.load_state_dict(state["optimizer_state"])
        sched.load_state_dict(state["scheduler_state"])
        best = state["best"]
        best_state = state["best_state"]
        bad = state["bad"]
        start_ep = state["next_epoch"]
        # Restore random states so resumed training follows the same random sequence.
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        torch.set_rng_state(state["torch_rng"])
        if torch.cuda.is_available() and state.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all(state["cuda_rng"])
        print(f"Resuming {task}/{name}/{mode}/seed={seed}/fold={fold} from epoch {start_ep+1}", flush=True)

    va_clin = clin_matrix(va, stats, np.zeros(len(va), bool), np.zeros(len(va), bool))
    for ep in range(start_ep, EPOCHS):
        model.train()
        order = np.random.choice(len(tr_in), len(tr_in), p=sample_p / sample_p.sum()) if sample_p is not None else np.random.permutation(len(tr_in))
        for i in range(0, len(order), BATCH):
            b = tr_in[order[i:i + BATCH]]
            if len(b) < 2: continue
            drop = np.random.rand(len(b)) < CLIN_DROP if use_clinical else np.zeros(len(b), bool)
            xb = augment(to_x(b)).to(device)
            cb = torch.tensor(clin_matrix(b, stats, drop, drop)).to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb, cb), torch.tensor(y[b]).to(device))
            loss.backward(); opt.step()
        sched.step()
        v = f1_score(y[va], predict(model, va, va_clin).argmax(1), average="macro")
        if v > best:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1

        # Save after EVERY epoch to Google Drive. If interrupted mid-epoch,
        # only that epoch may need to be repeated.
        atomic_torch_save({
            "model_state": model.state_dict(),
            "optimizer_state": opt.state_dict(),
            "scheduler_state": sched.state_dict(),
            "best_state": best_state,
            "best": best, "bad": bad, "next_epoch": ep + 1,
            "python_rng": random.getstate(),
            "numpy_rng": np.random.get_state(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }, ckpt)
        print(f"epoch {ep+1:02d}/{EPOCHS} | val macro-F1={v:.4f} | best={best:.4f} | checkpoint saved", flush=True)
        if bad >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(f"No best model state available for {task}/{name}/{mode}/seed={seed}/fold={fold}")
    model.load_state_dict(best_state)
    return model, stats

def run_study():
    t0 = time.time()
    if os.path.exists(PROGRESS_CSV):
        rows_df = pd.read_csv(PROGRESS_CSV)
        rows = rows_df.to_dict("records")
        print(f"Loaded {len(rows)} saved result rows; completed folds will be skipped.")
    else:
        rows = []

    for task in TASKS:
        y = make_labels(task)
        for name in MODELS:
            for mode, use_clin in [("MRI only", False), ("MRI + clinical", True)]:
                for seed in SEEDS:
                    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed)
                    for fold, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y)):
                        scenarios = MISSING_SCENARIOS if use_clin else {"0% missing": 0.0}
                        expected = len(scenarios)
                        already = [r for r in rows if r.get("task")==task and r.get("model")==name and
                                   r.get("mode")==mode and int(r.get("seed",-1))==seed and
                                   int(r.get("fold",-1))==fold]
                        if len(already) >= expected:
                            print(f"SKIP completed: {task} | {name} | {mode} | seed {seed} | fold {fold}", flush=True)
                            continue

                        model, stats = train_one(name, y, tr, use_clin, seed, task, mode, fold)
                        for sc, spec in scenarios.items():
                            # Do not duplicate scenarios if a partial result file already has them.
                            exists = any(r.get("task")==task and r.get("model")==name and r.get("mode")==mode and
                                         int(r.get("seed",-1))==seed and int(r.get("fold",-1))==fold and r.get("scenario")==sc
                                         for r in rows)
                            if exists: continue
                            rng = np.random.default_rng(seed * 100 + fold)
                            if spec == "age":
                                ha, hs = np.ones(len(te), bool), np.zeros(len(te), bool)
                            elif spec == "sex":
                                ha, hs = np.zeros(len(te), bool), np.ones(len(te), bool)
                            else:
                                ha = hs = rng.random(len(te)) < spec
                            p = predict(model, te, clin_matrix(te, stats, ha, hs))
                            rows.append(dict(task=task, model=name, mode=mode, scenario=sc, seed=seed, fold=fold, **score(y[te], p, task)))
                            # Save results after each scenario, not only at the end.
                            pd.DataFrame(rows).to_csv(PROGRESS_CSV, index=False)
                        print(f"{task} | {name} | {mode} | seed {seed} fold {fold} done | {time.time()-t0:.0f}s", flush=True)
                        # Once all fold metrics are saved, remove the epoch checkpoint.
                        cp = checkpoint_path(task, name, mode, seed, fold)
                        if os.path.exists(cp): os.remove(cp)
    results = pd.DataFrame(rows)
    results.to_csv(RAW_CSV, index=False)
    print("Saved all raw results to:", RAW_CSV)
    return results


In [ ]:
# ===== 5) RUN, SUMMARISE, SAVE =====
results = run_study()
cols = ["mcc", "macro_f1", "balanced_acc", "auc", "accuracy"]
g = results.groupby(["task", "model", "mode", "scenario"])[cols]
summary = (g.mean().round(3).astype(str) + " ± " + g.std().round(3).astype(str))
summary.to_csv(SUMMARY_CSV)
try:
    summary.to_excel(SUMMARY_XLSX)
except Exception as e:
    print("Excel not saved:", e)
for task in TASKS:
    print(f"\n### {task}: full data (0% missing)")
    try: print(summary.xs(task, level="task").xs("0% missing", level="scenario"))
    except KeyError: pass
    print(f"\n### {task}: MRI + clinical when clinical data are hidden")
    try: print(summary.xs(task, level="task").xs("MRI + clinical", level="mode")["macro_f1" if task == "3class" else "mcc"].unstack("scenario"))
    except KeyError: pass
print("\nAll results/checkpoints are saved in:", SAVE_DIR)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /content/drive/MyDrive/UCSF_PDGM_checkpoints/torch_hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 98.0MB/s]


epoch 01/20 | val macro-F1=0.1667 | best=0.1667 | checkpoint saved
epoch 02/20 | val macro-F1=0.4444 | best=0.4444 | checkpoint saved
epoch 03/20 | val macro-F1=0.4444 | best=0.4444 | checkpoint saved
epoch 04/20 | val macro-F1=0.6394 | best=0.6394 | checkpoint saved
epoch 05/20 | val macro-F1=0.6561 | best=0.6561 | checkpoint saved
epoch 06/20 | val macro-F1=0.5810 | best=0.6561 | checkpoint saved
epoch 07/20 | val macro-F1=0.7173 | best=0.7173 | checkpoint saved
epoch 08/20 | val macro-F1=0.6561 | best=0.7173 | checkpoint saved
epoch 09/20 | val macro-F1=0.7396 | best=0.7396 | checkpoint saved
epoch 10/20 | val macro-F1=0.6231 | best=0.7396 | checkpoint saved
epoch 11/20 | val macro-F1=0.6231 | best=0.7396 | checkpoint saved
epoch 12/20 | val macro-F1=0.7386 | best=0.7396 | checkpoint saved
epoch 13/20 | val macro-F1=0.5957 | best=0.7396 | checkpoint saved
epoch 14/20 | val macro-F1=0.5255 | best=0.7396 | checkpoint saved
binary | mobilenet_v3_small | MRI only | seed 42 fold 0 done |

100%|██████████| 20.5M/20.5M [00:00<00:00, 80.1MB/s]


epoch 01/20 | val macro-F1=0.2397 | best=0.2397 | checkpoint saved
epoch 02/20 | val macro-F1=0.4393 | best=0.4393 | checkpoint saved
epoch 03/20 | val macro-F1=0.7000 | best=0.7000 | checkpoint saved
epoch 04/20 | val macro-F1=0.7283 | best=0.7283 | checkpoint saved
epoch 05/20 | val macro-F1=0.7782 | best=0.7782 | checkpoint saved
epoch 06/20 | val macro-F1=0.7869 | best=0.7869 | checkpoint saved
epoch 07/20 | val macro-F1=0.7948 | best=0.7948 | checkpoint saved
epoch 08/20 | val macro-F1=0.7443 | best=0.7948 | checkpoint saved
epoch 09/20 | val macro-F1=0.7869 | best=0.7948 | checkpoint saved
epoch 10/20 | val macro-F1=0.8036 | best=0.8036 | checkpoint saved
epoch 11/20 | val macro-F1=0.8478 | best=0.8478 | checkpoint saved
epoch 12/20 | val macro-F1=0.7545 | best=0.8478 | checkpoint saved
epoch 13/20 | val macro-F1=0.7954 | best=0.8478 | checkpoint saved
epoch 14/20 | val macro-F1=0.7545 | best=0.8478 | checkpoint saved
epoch 15/20 | val macro-F1=0.7499 | best=0.8478 | checkpoint s

100%|██████████| 44.7M/44.7M [00:00<00:00, 118MB/s]


epoch 01/20 | val macro-F1=0.1894 | best=0.1894 | checkpoint saved
epoch 02/20 | val macro-F1=0.5255 | best=0.5255 | checkpoint saved
epoch 03/20 | val macro-F1=0.7954 | best=0.7954 | checkpoint saved
epoch 04/20 | val macro-F1=0.7869 | best=0.7954 | checkpoint saved
epoch 05/20 | val macro-F1=0.7304 | best=0.7954 | checkpoint saved
epoch 06/20 | val macro-F1=0.7948 | best=0.7954 | checkpoint saved
epoch 07/20 | val macro-F1=0.7221 | best=0.7954 | checkpoint saved
epoch 08/20 | val macro-F1=0.8327 | best=0.8327 | checkpoint saved
epoch 09/20 | val macro-F1=0.7948 | best=0.8327 | checkpoint saved
epoch 10/20 | val macro-F1=0.8478 | best=0.8478 | checkpoint saved
epoch 11/20 | val macro-F1=0.8667 | best=0.8667 | checkpoint saved
epoch 12/20 | val macro-F1=0.8478 | best=0.8667 | checkpoint saved
epoch 13/20 | val macro-F1=0.8408 | best=0.8667 | checkpoint saved
epoch 14/20 | val macro-F1=0.8408 | best=0.8667 | checkpoint saved
epoch 15/20 | val macro-F1=0.8478 | best=0.8667 | checkpoint s